In [ ]:
import dash
import dash_core_components as dcc
import dash_html_components as html
import pandas as pd
import numpy as np
import re
import plotly_express as px
from dash.dependencies import Input, Output
rootfold = '/home/j0hndoe/Documents/git/reddit-disinformation'

In [ ]:
def get_flag_perc(df, groups):
    return df.\
        groupby(groups)\
            [['n_subm_kept','ISFLAG']].\
                agg(sum).sort_index().reset_index().\
        assign(perc_FLAG = lambda df: df.ISFLAG/df.n_subm_kept)

In [ ]:
df_flags = pd.read_csv(rootfold+"/dashboard/data/app_flags_day.csv")
df_subm = pd.read_csv(rootfold+"/dashboard/data/app_subr_week.csv")

df_subm['week'] = df_subm['week'].astype('int')
min_week, max_week = df_subm['week'].min(), df_subm['week'].max()
weekrange = [min_week, max_week]
weekrange = [15, 29]

filtered_df_subm = df_day[(df_subm['week'].between(*weekrange))]# & (df['varr']==varr)]
filtered_df_subm['ISFLAG'] = filtered_df_subm.flag.copy()#loc[:,flagtypes].sum(axis=1)>0
### Aggregate
filtered_df_week = get_flag_perc(
    filtered_df_subm,
    groups = ['subreddit_cat','subreddit','week'])
filtered_df_agg = get_flag_perc(
    filtered_df_subm,
    groups = ['subreddit_cat','subreddit'])

In [ ]:
from datetime import datetime
x = "2020-01-05"
def revdate(datestring, year = False):
    if year:
        return datetime.strptime(datestring, "%Y-%m-%d").strftime("%m %d, %Y")
    else:
        return datetime.strptime(datestring, "%Y-%m-%d").strftime("%m %d")
datetime.strptime(x, "%Y-%m-%d").strftime("%b %d, %Y")
revdate(x, year=True)

In [ ]:
week_day_df = df_flags.groupby('week')['day'].agg(['min','max'])
week_day_df['label'] = [r[1]['min'][8:10] + '/' + r[1]['min'][5:7] + ' - ' + r[1]['max'][8:10] + '/' + r[1]['max'][5:7]
                        for r in week_day_df.iterrows()]
week_day_df['label'] = [r[1]['min'][5:7] + '/\n' + r[1]['min'][8:10]
                        for r in week_day_df.iterrows()]
week_day_df

In [ ]:
filtered_df_week_cat = get_flag_perc(filtered_df_week, ['subreddit_cat','week'])
tickvals0 = list(range(filtered_df_week_cat.week.min(),
           filtered_df_week_cat.week.max(),
           1)) + [filtered_df_week_cat.week.max()]
tickvals0 = subr_line_all.data[0].x
ticklabs0 = week_day_df['label'][tickvals0]
subr_line_all = px.line(
                    filtered_df_week_cat,
                    x = "week",
                    y = "ISFLAG",
                    color = "subreddit_cat")

subr_line_all.show()

In [ ]:
rng0 = [1,10]
def get_xlabs(xrange, nticks =5):    
    rng = xrange[1] - xrange[0]
    ticklst = list(range(xrange[0], xrange[1], rng // nticks+1)) 
    return ticklst + [xrange[1]]
    
get_xlabs(rng0)

In [ ]:
min([min(a.x) for a in subr_line_all.data])

In [ ]:

subr_line_all.update_layout(xaxis = dict(tickmode = 'array',
                                         tickvals = tickvals0,
                                         ticktext = ticklabs0
                                         ))

In [ ]:
dir(subr_line_all)

In [ ]:
subr_line_all.data[0].x

In [ ]:
fig_subr_treemap = px.treemap(filtered_df_agg, 
                      path=['subreddit_cat', 'subreddit'], 
                      values='ISFLAG',
                      color='subreddit_cat')
fig_subr_treemap.show()

### Cluster analysis

In [ ]:
subm = pd.read_csv(rootfold+"/dashboard/data/app_subm_day.csv")

In [ ]:
from sklearn.cluster import MiniBatchKMeans
flagtypes = list(subm.loc[:,"disinformation":"other"].columns)
nclust = 5

In [ ]:
filtered = subm.copy()        
filtered = filtered[(filtered['week'].between(*weekrange))]
filtered['ISFLAG'] = filtered[flagtypes].copy().sum(axis=1)
filtered = filtered[filtered['ISFLAG']>0]
filtered = filtered[filtered['subm_covid']]
### Run cluster analysis
if nclust>1:
    X = filtered[['PC01', 'PC02', 'PC03', 'PC04', 'PC05']]
    cluster = MiniBatchKMeans(n_clusters=nclust, batch_size=1000)
    cluster.fit(X)
    filtered['cluster'] = (cluster.predict(X)+1).astype(str)
#filtered

In [ ]:
cluster.cluster_centers_

In [ ]:
np.inner(X,cluster.cluster_centers_[0,:])

In [ ]:
np.inner(X,cluster.cluster_centers_[0,:])[5022]

In [ ]:
filtered[['subm_title','subreddit']].iloc[sel_submissions_index]

In [ ]:
sel_submissions_index = np.argsort(np.inner(X,cluster.cluster_centers_[0,:]))[-5:]
list(sel_submissions_index)
filtered.subm_title.iloc[sel_submissions_index]

In [ ]:
dotprod = X.dot(cluster.cluster_centers_.transpose())
dotprod.columns = ['C' + str(i+1) for i in range(nclust)]
joined = filtered[['subm_title','cluster']].join(dotprod)
for cl in range(nclust):
    print(dotprod.columns[cl])
    print(joined.sort_values(dotprod.columns[cl], ascending=False).subm_title.unique()[:20])

In [ ]:
joined.head()

In [ ]:
[{'label': 'Cluster '+str(i), 'value': i} for i in range(1,nclust+1)]

#### TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidfer = TfidfVectorizer(min_df=5, stop_words='english')
tf_idf_matrix = tfidfer.fit_transform(joined.subm_title)
feature_names = np.array(tfidfer.get_feature_names())
tf_idf_matrix

In [ ]:
tf_idf_matrix[:10,:10].data

In [ ]:
indices = np.argsort(tfidfer.idf_)[::-1]
features = tfidfer.get_feature_names()
top_n = 20
top_features = [features[i] for i in indices[:top_n]]
print(top_features)

In [ ]:
def top_tfidf_feats(row, features, top_n=25):
    ''' Get top n tfidf values in row and return them with their corresponding feature names.'''
    topn_ids = np.argsort(row)[::-1][:top_n]
    top_feats = [(features[i], row[i]) for i in topn_ids]
    df = pd.DataFrame(top_feats)
    df.columns = ['feature', 'tfidf']
    return df
def top_mean_feats(Xtr, features, grp_ids=None, min_tfidf=0.1, top_n=25):
    ''' Return the top n features that on average are most important amongst documents in rows
        indentified by indices in grp_ids. '''
    if grp_ids:
        D = Xtr[grp_ids].toarray()
    else:
        D = Xtr.toarray()

    D[D < min_tfidf] = 0
    tfidf_means = np.mean(D, axis=0)
    return top_tfidf_feats(tfidf_means, features, top_n)
def top_feats_by_class(Xtr, y, features, min_tfidf=0.1, top_n=25):
    ''' Return a list of dfs, where each df holds top_n features and their mean tfidf value
        calculated across documents with the same class label. '''
    dfs = []
    labels = np.unique(y)
    for label in labels:
        ids = np.where(y==label)
        feats_df = top_mean_feats(Xtr, features, ids, min_tfidf=min_tfidf, top_n=top_n)
        feats_df.label = label
        print('\nCLUSTER '+label)
        print(feats_df)
        #return dfs

In [ ]:
features = tfidfer.get_feature_names()
top_feats_by_class(tf_idf_matrix, joined.cluster, features, min_tfidf=0.1, top_n=10)

In [ ]:
from wordcloud import WordCloud

In [ ]:
wordcloud = WordCloud (
                    background_color = 'white',
                    width = 512,
                    height = 384
                        ).generate(' '.join(joined.subm_title))
plt.imshow(wordcloud) # image show
plt.axis('off') # to off the axis of x and y

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS
from collections import Counter
def cluster_commonwords(texts, nwords = 5):
    allwords = [w for w in ' '.join(texts.str.lower()).split() if w not in STOP_WORDS and re.search('[a-z]',w)]
    return ', '.join([word for word,cnt in Counter(allwords).most_common(nwords)])

In [ ]:
c = cluster_commonwords(joined.subm_title, nwords = 10)

In [ ]:
STOP_WORDS.union(['coronavirus','covid','covid19','covid-19'])

In [ ]:
def generate_table(dataframe, max_rows=10):
    return html.Table(
        # Header
        [html.Tr([html.Th(col) for col in dataframe.columns])] +

        # Body
        [html.Tr([
            html.Td(dataframe.iloc[i][col]) for col in dataframe.columns
        ]) for i in range(min(len(dataframe), max_rows))]
    )


In [ ]:
generate_table(joined)